# 📄 Data Wrangling & EDA — CV Classification (NLP)
> **Capstone Project — Data Scientist**  
> Dataset: `Dataset_NLP_Siap_Model.xlsx` — 2.241 CV dari 8 kategori jabatan

---
**Alur kerja:**
1. Gathering Data
2. Assessing Data
3. Cleaning Data
4. EDA (Exploratory Data Analysis)

## 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

print('Library berhasil di-import!')

---
## 1. Gathering Data
Memuat dataset yang sudah diproses dari tahap ekstraksi teks (PDF → Excel).

In [ ]:
# Load dataset utama
df = pd.read_excel('../data/processed/Dataset_NLP_Siap_Model.xlsx')

print(f'✅ Dataset berhasil dimuat!')
print(f'   Jumlah baris : {df.shape[0]:,}')
print(f'   Jumlah kolom : {df.shape[1]}')
print(f'   Kolom        : {list(df.columns)}')
print()
df.head()

---
## 2. Assessing Data
Mengevaluasi kualitas data — mencari missing values, duplikat, tipe data yang salah, dan anomali.

In [ ]:
# ── 2.1 Informasi Umum ──
print('=== INFO DATAFRAME ===')
df.info()

In [ ]:
# ── 2.2 Statistik Deskriptif ──
print('=== STATISTIK DESKRIPTIF ===')
df.describe(include='all')

In [ ]:
# ── 2.3 Cek Missing Values ──
missing = df.isnull().sum()
pct_missing = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': pct_missing
})

print('=== CEK MISSING VALUES ===')
print(missing_df)

if missing.sum() == 0:
    print('\n✅ Tidak ada missing values!')
else:
    print(f'\n⚠️  Total missing values: {missing.sum()}')

In [ ]:
# ── 2.4 Cek Duplikasi ──
n_dup_row  = df.duplicated().sum()
n_dup_id   = df['ID_CV'].duplicated().sum()
n_dup_text = df['Clean_Text'].duplicated().sum()

print('=== CEK DUPLIKASI ===')
print(f'Baris duplikat (seluruh kolom) : {n_dup_row}')
print(f'ID_CV duplikat                 : {n_dup_id}')
print(f'Clean_Text duplikat            : {n_dup_text}')

if n_dup_text > 0:
    print('\n⚠️  Beberapa teks CV identik:')
    print(df[df['Clean_Text'].duplicated(keep=False)][['ID_CV', 'Category']].sort_values('Category'))

In [ ]:
# ── 2.5 Distribusi Kategori ──
print('=== DISTRIBUSI KATEGORI ===')
cat_counts = df['Category'].value_counts()
print(cat_counts)
print(f'\nTotal kategori unik: {df["Category"].nunique()}')

In [ ]:
# ── 2.6 Cek Anomali Word Count ──
print('=== ANALISIS WORD_COUNT ===')
print(df['Word_Count'].describe())

# Deteksi outlier dengan metode IQR
q1 = df['Word_Count'].quantile(0.25)
q3 = df['Word_Count'].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas  = q3 + 1.5 * iqr

outlier_bawah = df[df['Word_Count'] < batas_bawah]
outlier_atas  = df[df['Word_Count'] > batas_atas]

print(f'\nBatas bawah IQR : {batas_bawah:.0f} kata')
print(f'Batas atas IQR  : {batas_atas:.0f} kata')
print(f'CV sangat pendek (< {batas_bawah:.0f} kata) : {len(outlier_bawah)} baris')
print(f'CV sangat panjang (> {batas_atas:.0f} kata) : {len(outlier_atas)} baris')

In [ ]:
# ── 2.7 Cek Format ID_CV ──
print('=== VALIDASI FORMAT ID_CV ===')
pattern_id = r'^CV_\d{4}$'
id_tidak_valid = df[~df['ID_CV'].str.match(pattern_id)]

if len(id_tidak_valid) == 0:
    print('✅ Semua ID_CV memiliki format yang benar (CV_XXXX)')
else:
    print(f'⚠️  {len(id_tidak_valid)} ID_CV tidak mengikuti format baku:')
    print(id_tidak_valid['ID_CV'].values)

In [ ]:
# ── 2.8 Ringkasan Temuan Assessment ──
cat_counts = df['Category'].value_counts()
print('=' * 50)
print('RINGKASAN TEMUAN DATA ASSESSMENT')
print('=' * 50)
print(f'1. Missing values      : Tidak ada ✅')
print(f'2. Baris duplikat      : {df.duplicated().sum()} ✅')
print(f'3. ID duplikat         : {df["ID_CV"].duplicated().sum()} ✅')
print(f'4. Tipe data           : Sesuai ✅')
print(f'5. Format ID_CV        : Valid ✅')
print(f'6. Outlier Word_Count  : {len(outlier_bawah) + len(outlier_atas)} baris ⚠️  (dipertahankan untuk NLP)')
print(f'7. Imbalance kategori  : Ada ⚠️  (React Developer: {cat_counts.min()} vs Business Analyst: {cat_counts.max()})')

---
## 3. Cleaning Data
Berdasarkan assessment, data sudah cukup bersih. Proses cleaning berfokus pada:
- Normalisasi kolom `Category`
- Validasi `Word_Count` vs panjang teks aktual
- Menambah kolom label numerik untuk pemodelan

In [ ]:
# ── 3.1 Normalisasi Category (strip whitespace) ──
df['Category'] = df['Category'].str.strip()

print('✅ Kolom Category sudah dinormalisasi.')
print('Kategori final:', sorted(df['Category'].unique().tolist()))

In [ ]:
# ── 3.2 Validasi Word Count vs Panjang Teks Aktual ──
df['Word_Count_Aktual'] = df['Clean_Text'].apply(lambda x: len(str(x).split()))
selisih = (df['Word_Count'] - df['Word_Count_Aktual']).abs()

print(f'Rata-rata selisih Word_Count vs hitungan ulang: {selisih.mean():.1f} kata')
print(f'Selisih maksimum: {selisih.max()} kata')
print('\n(Perbedaan wajar akibat perbedaan metode tokenisasi)')

# Gunakan Word_Count original
df.drop(columns='Word_Count_Aktual', inplace=True)
print('\n✅ Word_Count original dipertahankan.')

In [ ]:
# ── 3.3 Encode Label Numerik ──
kategori_urut = sorted(df['Category'].unique())
label_map = {cat: idx for idx, cat in enumerate(kategori_urut)}

df['Label'] = df['Category'].map(label_map)

print('Mapping Kategori → Label Numerik:')
for k, v in label_map.items():
    print(f'  {v}  →  {k}')

print(f'\n✅ Kolom Label berhasil ditambahkan.')

In [ ]:
# ── 3.4 Tambah Kolom Char_Count (Fitur Tambahan) ──
df['Char_Count'] = df['Clean_Text'].apply(lambda x: len(str(x)))

print('✅ Kolom Char_Count berhasil ditambahkan.')
print()
print('Preview dataset setelah cleaning:')
df[['ID_CV', 'Category', 'Label', 'Word_Count', 'Char_Count']].head()

In [ ]:
# ── 3.5 Dataset Final ──
print('=== DATASET SETELAH CLEANING ===')
print(f'Shape : {df.shape}')
print(f'Kolom : {list(df.columns)}')
print()
df.head()

---
## 4. Exploratory Data Analysis (EDA)
Memahami pola dan karakteristik data melalui visualisasi.

In [ ]:
# ── 4.1 Distribusi CV per Kategori ──
os.makedirs('../reports', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cat_counts_sorted = df['Category'].value_counts().sort_values(ascending=True)
colors = sns.color_palette('Set2', len(cat_counts_sorted))

# Bar chart horizontal
bars = axes[0].barh(cat_counts_sorted.index, cat_counts_sorted.values, color=colors)
axes[0].set_title('Jumlah CV per Kategori Jabatan', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Jumlah CV')
for bar, val in zip(bars, cat_counts_sorted.values):
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 f'{val}', va='center', fontsize=10)

# Pie chart
axes[1].pie(cat_counts_sorted.values, labels=cat_counts_sorted.index,
            autopct='%1.1f%%', startangle=90, colors=colors)
axes[1].set_title('Proporsi CV per Kategori', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/01_distribusi_kategori.png', bbox_inches='tight')
plt.show()

print('📌 Insight: Distribusi tidak seimbang (class imbalance).')
print('   React Developer hanya 159 CV, Business Analyst mencapai 314 CV.')
print('   Perlu strategi oversampling / class_weight saat pemodelan.')

In [ ]:
# ── 4.2 Distribusi Word Count ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(df['Word_Count'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Word_Count'].mean(),   color='red',    linestyle='--', label=f'Mean  : {df["Word_Count"].mean():.0f}')
axes[0].axvline(df['Word_Count'].median(), color='orange', linestyle='--', label=f'Median: {df["Word_Count"].median():.0f}')
axes[0].set_title('Distribusi Jumlah Kata CV', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Boxplot per kategori
order_cat = df.groupby('Category')['Word_Count'].median().sort_values().index
sns.boxplot(data=df, y='Category', x='Word_Count', order=order_cat,
            palette='Set2', ax=axes[1])
axes[1].set_title('Distribusi Word Count per Kategori', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/02_distribusi_wordcount.png', bbox_inches='tight')
plt.show()

print('📌 Insight: Distribusi word count right-skewed (ekor panjang ke kanan).')
print('   SAP & React Developer cenderung memiliki CV yang lebih panjang.')

In [ ]:
# ── 4.3 Statistik Word Count per Kategori ──
wc_stats = df.groupby('Category')['Word_Count'].agg(['mean','median','min','max','std'])
wc_stats.columns = ['Rata-rata', 'Median', 'Min', 'Maks', 'Std Dev']
wc_stats = wc_stats.round(1).sort_values('Median', ascending=False)

print('=== STATISTIK WORD COUNT PER KATEGORI ===')
display(wc_stats)

In [ ]:
# ── 4.4 Visualisasi Outlier Word Count ──
q1 = df['Word_Count'].quantile(0.25)
q3 = df['Word_Count'].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas  = q3 + 1.5 * iqr
outlier = df[(df['Word_Count'] < batas_bawah) | (df['Word_Count'] > batas_atas)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(df.index, df['Word_Count'], alpha=0.35, color='steelblue', s=10, label='Normal')
ax.scatter(outlier.index, outlier['Word_Count'], color='red', s=25,
           label=f'Outlier ({len(outlier)} CV)', zorder=5)
ax.axhline(batas_atas,  color='red',    linestyle='--', alpha=0.7, label=f'Batas atas : {batas_atas:.0f}')
ax.axhline(batas_bawah, color='orange', linestyle='--', alpha=0.7, label=f'Batas bawah: {batas_bawah:.0f}')
ax.set_title('Scatter Plot Word Count — Deteksi Outlier (IQR Method)', fontsize=14, fontweight='bold')
ax.set_xlabel('Index CV')
ax.set_ylabel('Jumlah Kata')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/03_outlier_wordcount.png', bbox_inches='tight')
plt.show()

print(f'📌 Ditemukan {len(outlier)} CV sebagai outlier berdasarkan metode IQR.')
print('   Untuk model NLP, outlier ini TIDAK dihapus — teks tetap valid dan informatif.')
print()
print('Distribusi outlier per kategori:')
print(outlier['Category'].value_counts())

In [ ]:
# ── 4.5 Korelasi Word Count vs Character Count ──
fig, ax = plt.subplots(figsize=(10, 6))

palette = sns.color_palette('Set2', df['Category'].nunique())
for i, (cat, group) in enumerate(df.groupby('Category')):
    ax.scatter(group['Word_Count'], group['Char_Count'],
               alpha=0.4, color=palette[i], label=cat, s=14)

ax.set_title('Korelasi Word Count vs Character Count per Kategori',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Character Count')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/04_wordcount_vs_charcount.png', bbox_inches='tight')
plt.show()

corr = df['Word_Count'].corr(df['Char_Count'])
print(f'📌 Korelasi Pearson Word Count & Char Count: {corr:.4f}')
print('   Korelasi sangat tinggi — kedua fitur mengukur hal yang sama (panjang teks).')

In [ ]:
# ── 4.6 Violin Plot — Distribusi Word Count Detail ──
fig, ax = plt.subplots(figsize=(14, 6))

order_cat = df.groupby('Category')['Word_Count'].median().sort_values().index
sns.violinplot(data=df, x='Category', y='Word_Count', order=order_cat,
               palette='Set2', inner='quartile', ax=ax)

ax.set_title('Violin Plot: Distribusi Word Count per Kategori',
             fontsize=14, fontweight='bold')
ax.set_xticklabels(order_cat, rotation=25, ha='right')
ax.set_xlabel('')
ax.set_ylabel('Jumlah Kata')
plt.tight_layout()
plt.savefig('../reports/05_violin_wordcount.png', bbox_inches='tight')
plt.show()

print('📌 Insight: Variasi panjang CV cukup tinggi di semua kategori.')
print('   Ini wajar — tidak ada standar baku panjang CV per profesi.')

In [ ]:
# ── 4.7 Heatmap Korelasi Fitur Numerik ──
numeric_cols = df[['Word_Count', 'Char_Count', 'Label']]
corr_matrix = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Heatmap Korelasi Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/06_heatmap_korelasi.png', bbox_inches='tight')
plt.show()

print('📌 Word_Count dan Char_Count sangat berkorelasi (redundan).')
print('   Dalam pemodelan, cukup gunakan salah satu sebagai fitur numerik tambahan.')

---
## 5. Ringkasan & Rekomendasi

### ✅ Temuan Utama:
| Aspek | Temuan |
|---|---|
| **Ukuran Dataset** | 2.241 CV dari 8 kategori jabatan |
| **Missing Values** | Tidak ada |
| **Duplikasi** | Tidak ada |
| **Tipe Data** | Semua kolom sudah sesuai |
| **Class Imbalance** | Ada — React Developer (159) vs Business Analyst (314) |
| **Outlier Word Count** | Ada, namun dipertahankan karena teks valid |
| **Korelasi Fitur** | Word_Count & Char_Count sangat berkorelasi (redundan) |

### 🎯 Rekomendasi untuk Pemodelan NLP:
1. **Gunakan kolom `Clean_Text`** sebagai input utama model (teks sudah bersih)
2. **Gunakan kolom `Label`** sebagai target klasifikasi (0–7)
3. **Atasi class imbalance** dengan `class_weight='balanced'` atau SMOTE
4. **Word_Count** bisa dijadikan fitur tambahan (numerik) untuk model hybrid
5. **Train/Val/Test split** disarankan 70:15:15 dengan stratifikasi per kategori

In [ ]:
# ── 5. Simpan Dataset Bersih Final ──
output_path = '../data/processed/Dataset_Wrangled.xlsx'
df.to_excel(output_path, index=False, engine='openpyxl')

print(f'✅ Dataset hasil wrangling disimpan ke: {output_path}')
print(f'   Shape final : {df.shape}')
print(f'   Kolom       : {list(df.columns)}')
print()
df.head(3)